In [ ]:
def is_valid_partition(p):
    """Check if p is a valid partition (decreasing sequence of positive integers)."""
    return all(p[i] >= p[i+1] > 0 for i in range(len(p) - 1))

def transpose_partition(p):
    """
    Compute the transpose of partition p.
    """
    if not p:
        return []
    return [sum(1 for part in p if part > i) for i in range(p[0])]

def hook_length(p, i, j):
    """
    Compute the hook length for the (i,j)-th element in partition p.
    i and j are 0-indexed.
    """
    if i >= len(p) or j >= p[i]:
        return 0
    arm = p[i] - j - 1
    leg = sum(1 for row in p[i+1:] if row > j)
    return arm + leg + 1

def getd(l,i,default=None):
    return l[i] if i < len(l) else default

def remove_hook(p, i, j):
    """
    Remove a hook with (i,j) at the corner in the partition p.
    Returns the new partition after hook removal.
    """
    hook_len = hook_length(p, i, j)
    new_p = p.copy()
    
    # Remove from the leg
    for k in range(i, len(new_p)):
        if new_p[k] > j:
            new_p[k] = j + max(getd(new_p,k+1,0)-j-1,0) 
        else:
            break
    
    # Remove any resulting zeros
    new_p = [part for part in new_p if part > 0]
    
    return new_p, hook_len

def to_e_core(p, e):
    """
    Compute the e-core of partition p.
    """
    if not is_valid_partition(p):
        raise ValueError("Invalid partition: Must be a decreasing sequence of positive integers")
    if e <= 0:
        raise ValueError("e must be a positive integer")

    p = p.copy()  # Create a copy to avoid modifying the original
    
    while True:
        hook_removed = False
        for i in range(len(p)):
            for j in range(p[i]):
                if hook_length(p, i, j) == e:
                    p, _ = remove_hook(p, i, j)
                    hook_removed = True
                    break
            if hook_removed:
                break
        if not hook_removed:
            break
    
    return p

def print_hook_lengths(p):
    """
    Print the hook lengths of a partition p in a diagram format.
    """
    if not p:
        print("Empty partition")
        return

    # Calculate the maximum width of hook length values for formatting
    max_width = len(str(max(hook_length(p, i, j) for i in range(len(p)) for j in range(p[i]))))

    for i in range(len(p)):
        row = ""
        for j in range(p[i]):
            hl = hook_length(p, i, j)
            row += f"{hl:^{max_width}} "
        print(row.rstrip())  # Remove trailing space
    print()  # Add a blank line after the diagram


def are_partitions_close(p1, p2):
    """
    Check if two partitions p1 and p2 are close.
    Two partitions are close if the absolute difference between
    corresponding parts is at most 1.

    :param p1: First partition (list of integers)
    :param p2: Second partition (list of integers)
    :return: Boolean indicating if the partitions are close
    """
    if not (is_valid_partition(p1) and is_valid_partition(p2)):
        raise ValueError("Invalid partition(s): Must be decreasing sequences of positive integers")

    # Make copies to avoid modifying the original partitions
    p1 = p1.copy()
    p2 = p2.copy()

    # Extend the shorter partition with zeros
    max_len = max(len(p1), len(p2))
    p1.extend([0] * (max_len - len(p1)))
    p2.extend([0] * (max_len - len(p2)))

    # Check if the absolute difference between corresponding parts is at most 1
    for i in range(max_len):
        if abs(p1[i] - p2[i]) > 1:
            return False

    return True

def is_partition_even(p):
    """
    Check if a partition is even.
    A partition is even if its transpose has only even parts.

    :param p: Partition (list of integers)
    :return: Boolean indicating if the partition is even
    """
    if not is_valid_partition(p):
        raise ValueError("Invalid partition: Must be a decreasing sequence of positive integers")

    # Compute the transpose
    transpose = transpose_partition(p)

    # Check if all parts in the transpose are even
    return all(part % 2 == 0 for part in transpose)

def partition_intersection(p1, p2):
    """
    Compute the intersection of two partitions p1 and p2.
    The intersection is defined as [μi] where μi = μ'i.

    :param p1: First partition (list of integers)
    :param p2: Second partition (list of integers)
    :return: A new partition representing the intersection
    """
    if not (is_valid_partition(p1) and is_valid_partition(p2)):
        raise ValueError("Invalid partition(s): Must be decreasing sequences of positive integers")

    # Compute the intersection
    intersection = []
    for i in range(min(len(p1), len(p2))):
        if p1[i] == p2[i]:
            intersection.append(p1[i])

    return intersection

def find_close_partitions(p, n, m=None):
    """
    Recursively generate all partitions of size n that are close to the given partition p,
    with maximal part not larger than m.
    
    :param p: The input partition (list of integers)
    :param n: The desired size of close partitions
    :param m: The maximum allowed part
    :yield: Partitions of size n close to p with maximal part <= m
    """
    # Recursive cases
    if len(p) > 0:
        # Try keep the first part
        if (m is None or p[0] <= m):
            for x in find_close_partitions(p[1:], n - p[0], p[0]):
                yield [p[0]] + x

        # Try decreasing the first part
        if p[0] > 1 and (m is None or p[0] - 1 <= m):
            for x in find_close_partitions(p[1:], n - (p[0] - 1), p[0] - 1):
                yield [p[0] - 1] + x

        # Try increasing the first part
        if  m is None or p[0] + 1 <= m:
            for x in find_close_partitions(p[1:], n - (p[0] + 1), p[0] + 1):
                yield [p[0] + 1] + x

        # Try removing the first part if it's 1
        if p[0] == 1:
            if n == 0:
                yield []
            # Otherwise no yields 

    else: 
        # Try keep the part
        if n == 0: 
            yield []
        # Try adding a new part
        if 1 <= n and (m is None or 1 <= m):
            for x in  find_close_partitions([], n-1, 1):
                yield [1] + x


def print_partition(p, label=None):
    """
    Print a partition in a visually appealing format.
    
    :param partition: List of integers representing the partition
    :param label: Optional label to print before the partition
    """
    if label:
        print(f"{label}:")
    for i in p:
        print("*"*i)

def print_partitions(partitions, label=None):
    """
    Print multiple partitions.
    
    :param partitions: List of partitions (each partition is a list of integers)
    :param label: Optional label to print before the list of partitions
    """
    if label:
        print(f"{label}:")
    for i,p in enumerate(partitions, 1):
        print_partition(p, f"Partition {i}")
    print()  # Add an empty line after all partitions

def generate_partitions(n, max_part=None, current_partition=None):
    """
    Generate all partitions of n.
    
    :param n: The integer to partition
    :param max_part: The maximum allowed part size (optional)
    :param current_partition: The current partition being built (used in recursion)
    :yield: Each partition as a list of integers
    """
    if current_partition is None:
        current_partition = []
    
    if n == 0:
        yield current_partition
    else:
        start = min(n, max_part or n)
        for i in range(start, 0, -1):
            new_partition = current_partition + [i]
            yield from generate_partitions(n - i, i, new_partition)


def theta_lift(p, n):
    """
    Compute the theta lift of a partition.
    
    :param partition: The input partition
    :param n: The integer being partitioned
    :return: A list of partitions in the theta lift

    these are all partitions of n which is 2-transverse to p
    """
    result = []
    for x in find_close_partitions(p,n):
        if is_partition_even(partition_intersection(p,x)): 
            result.append(x)
    return result


In [ ]:
p = [7, 5, 3, 2, 1]
e = 4

print(f"Original partition: {p}")
print(f"Transpose of partition: {transpose_partition(p)}")
print(f"Hook length of (0,0): {hook_length(p, 0, 0)}")
print(f"e value: {e}")
print("\nHook lengths of the partition:")
print_hook_lengths(p)

e_core = to_e_core(p, e)

print(f"\nthe {e}-core")
print_hook_lengths(e_core)


print(f"{e}-core of the partition: {e_core}")

def print_all_partitions(n):
    """
    Generate and print all partitions of n.
    
    :param n: The integer to partition
    """
    partitions = list(generate_partitions(n))
    print_partitions(partitions, f"All partitions of {n}")
    print(f"Total number of partitions: {len(partitions)}")


In [ ]:
# Example usage
p = [4, 2, 1]
n = 7
m = 4

p = [2,2]

print_partition(p, "Original partition")

close_partitions = list(find_close_partitions(p, n, None))
print_partitions(close_partitions, f"Close partitions (n={n}, m={m})")
print_all_partitions(5)


In [ ]:
def test_theta_lift_and_e_cores(m, n, e):
    """
    Generate all partitions of m, compute their theta lifts to n,
    and calculate the corresponding e-cores.

    :param m: The integer to generate partitions from
    :param n: The integer to lift partitions to
    :param e: The e value for e-core calculation
    """
    all_partitions = list(generate_partitions(m))
    
    print(f"Testing theta lifts from partitions of {m} to {n}, with e-cores (e={e}):")
    print("=" * 60)

    for idx, partition in enumerate(all_partitions, 1):
        print(f"\nPartition {idx} of {m}: {partition} with {e}-core {to_e_core(partition,e)}")
        print_partition(partition, "Visual representation")

        lift = theta_lift(partition, n)
        print(f"Theta lift to {n} (count: {len(lift)}):")
        for lifted_partition in lift:
            e_core = to_e_core(lifted_partition, e)
            print(f"  {lifted_partition} -> e-core: {e_core}")

        print("-" * 40)

    print("\nSummary of unique e-cores:")
    all_e_cores = set()
    for partition in all_partitions:
        lift = theta_lift(partition, n)
        for lifted_partition in lift:
            all_e_cores.add(tuple(to_e_core(lifted_partition, e)))
    
    for idx, e_core in enumerate(sorted(all_e_cores), 1):
        print(f"{e}-core {idx}: {list(e_core)}")

# Example usage
m, n, e = 4, 6, 4
test_theta_lift_and_e_cores(m, n, e)